# 1. Overview and Setup

This notebook contains the code used for the network analysis discussed in the accompanying paper.

The aim of the project is to examine whether Empirical Network Analysis (ENA) can support a general account of the diffusion of scientific innovations. To do so, I apply a procedure similar to the one used by Herfeld and Doehne for rational choice theory to a different case study: the diffusion of Shannon’s information theory.

The collection and preprocessing of the bibliographic data were carried out separately, following the procedure described in the paper. The notebook therefore starts from two CSV files: one containing the nodes of the network and their metadata, and one containing the edges.

The notebook is organized as follows:

- **Section 1** loads the data and builds the graph with NetworkX.
- **Section 2** identifies the epistemic core, first by k-core decomposition and then by an alternative procedure based on degree centrality.
- **Section 3** identifies communities with the Girvan–Newman algorithm and compares the result with a Louvain partition using the Adjusted Rand Index (ARI).
- **Section 4** identifies the roles of elaborators and translators on the basis of the analytical criteria discussed in the paper.

This notebook is meant to document the computational analysis. The philosophical motivation, the justification of the methodological choices, and the interpretation of the results are given in the paper.


## 1.1 Upload the Data

In [1]:
import pandas as pd
from pathlib import Path


nodes_path = Path("data/nodes.csv")
edges_path = Path("data/edges.csv")

if not nodes_path.exists() or not edges_path.exists():
    raise FileNotFoundError("Missing input files: data/nodes.csv and/or data/edges.csv")

nodes_df = pd.read_csv(nodes_path)
edges_df = pd.read_csv(edges_path)



nodes_df.head()

,node_id,reference_occurrence,degree,year,doi,title,cited_by_count,authors
0,https://openalex.org/W2069271851,3,1,1972,10.1137/1.9781611970654.ch1,1. Bayesian Statistics a Review,530,D. V. Lindley
1,https://openalex.org/W2148962857,10,1,1955,10.2307/1884852,A Behavioral Model of Rational Choice,14887,Herbert A. Simon
2,https://openalex.org/W1568559199,3,1,1970,10.1007/bf00533669,A characterization of limiting distributions o...,278,Jaroslav H�jek
3,https://openalex.org/W2117476053,3,1,1956,10.1002/j.1538-7305.1956.tb02379.x,A Class of Binary Signaling Alphabets,195,D. Slepian
4,https://openalex.org/W1973071586,10,2,1972,10.1007/bf02018661,A class of measures of informativity of observ...,195,Imre Csiszár


A note on the attributes:

- **reference_occurence** indicates how many times the publication was cited in the publications that also cite Shannon’s *A Mathematical Theory of Communication*.
- **degree** is the number of other publications to which a publication is connected in the network.
- **cited_by_count** is the overall citation count of the publication in the broader bibliographic record.

## 1.2 Graph with NetworkX

In [2]:
import networkx as nx


# here the graph is created, using the CSV files. attributes are also added to the nodes

G = nx.Graph()


for _, row in nodes_df.iterrows():
    node_id = row["node_id"]
    attrs = row.drop("node_id").to_dict()
    G.add_node(node_id, **attrs)

for _, row in edges_df.iterrows():
    source = row["source"]
    target = row["target"]
    G.add_edge(source, target)

print(G.number_of_nodes(), "nodes")
print(G.number_of_edges(), "edges")


659 nodes
1048 edges


## 1.3 Plot

In [3]:
import plotly.graph_objects as go



pos = nx.spring_layout(G, k=0.15, iterations=50, seed=42)

def plot_graph(G, title = None): #the function plots the graph using plotly

  edge_x = []
  edge_y = []

  for u, v in G.edges():
      x0, y0 = pos[u]
      x1, y1 = pos[v]
      edge_x.extend([x0, x1, None])
      edge_y.extend([y0, y1, None])

  edge_trace = go.Scatter(
      x=edge_x,
      y=edge_y,
      mode="lines",
      line=dict(width=0.5, color="rgba(150,150,150,0.4)"),
      hoverinfo="skip"
  )


  node_x = []
  node_y = []
  hover_text = []
  node_size = []

  for n, attrs in G.nodes(data=True):
      x, y = pos[n]
      node_x.append(x)
      node_y.append(y)


      txt = f"<b>ID</b>: {n}<br>"
      for k, v in attrs.items():
          txt += f"<b>{k}</b>: {v}<br>"
      hover_text.append(txt)


      s = attrs.get("size", 8)
      try:
          s = float(s)
      except:
          s = 8
      node_size.append(s)

  node_trace = go.Scatter(
      x=node_x,
      y=node_y,
      mode="markers",
      hoverinfo="text",
      text=hover_text,
      marker=dict(
          size=node_size,
          sizemode="diameter",
          line=dict(width=0.5, color="white")
      )
  )


  fig = go.Figure(data=[edge_trace, node_trace])

  fig.update_layout(
      title= title,
      showlegend=False,
      hovermode="closest",
      margin=dict(l=10, r=10, t=40, b=10),
      xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
      yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
      height=900
  )

  fig.show()

plot_graph(G, title = "Spread of Information Theory")

# 2. Epistemic Core

This section identifies the epistemic core of the network.

In Herfeld and Doehne’s approach, the epistemic core is the “backbone” of the epistemic domain: conceptually, it consists of those publications that engage with the innovation early on, develop it, and make it suitable for application across different fields. Analytically, they identify it by means of k-core decomposition.

I follow that procedure first, and then consider an alternative approach based on degree centrality. The comparison is relevant because one of the questions raised in the paper is whether the identification of the epistemic core is robust with respect to different analytical choices.

## 2.1 Epistemic Core with K-core Decomposition



In [4]:
#k-core decomposition--> there is a built-in function in NetworkX

for n in range (0, 12):

  G_core = nx.k_core(G, k=n)

  print("K =", n, "nodes=", G_core.number_of_nodes())


K = 0 nodes= 659
K = 1 nodes= 659
K = 2 nodes= 309
K = 3 nodes= 180
K = 4 nodes= 94
K = 5 nodes= 40
K = 6 nodes= 23
K = 7 nodes= 23
K = 8 nodes= 22
K = 9 nodes= 15
K = 10 nodes= 11
K = 11 nodes= 0


First, I try with **k = 7**, in order to see whether the result is comparable to the case studied by Herfeld and Doehne.

However, the resulting subgraph does not seem to be a good candidate for the epistemic core. Rather than capturing a broadly interdisciplinary backbone of the diffusion of information theory, it mainly selects publications belonging to a more specialized area, especially cognitive science. For that reason, I do not retain it as the epistemic core.

In [5]:
G_core = nx.k_core(G, k=7)
plot_graph(G_core, title = "7-core")

I therefore lower the value to **k = 4** and select the largest connected component of the resulting k-core.

This choice is motivated by the fact that higher values of k yield subgraphs that appear too specialized, whereas the largest connected component of the 4-core includes publications that are more plausibly part of the broader epistemic backbone of the innovation.

In [6]:
G_core = nx.k_core(G, k=4)
core_nodes= max(nx.connected_components(G_core), key=len)
G_core = G_core.subgraph(core_nodes).copy()

plot_graph(G_core, title = "4-core, largest component")
print(" \n Number of members of the epistemic core:", G_core.number_of_nodes())


 
 Number of members of the epistemic core: 53


In [7]:
#here I visualize the metadata for the nodes in the core. I did that to make sure that this subgraph could indeed be an acceptable candidate as the epistemic core.

df_core = nodes_df[nodes_df["node_id"].isin(G_core.nodes())]
df_core

,node_id,reference_occurrence,degree,year,doi,title,cited_by_count,authors
6,https://openalex.org/W2057178439,8,5,1973,10.1287/mnsc.19.8.936,A Class of Solutions for Group Decision Problems,1090,P. L. Yu
9,https://openalex.org/W2001935257,14,8,1965,10.1016/s0019-9958(65)90332-3,A coding theorem and Rényi's entropy,276,L. L. Campbell
17,https://openalex.org/W2095087262,5,4,1969,10.1111/j.1529-8817.1969.tb02581.x,A COMPOSITE RATING OF ALGAE TOLERATING ORGANIC...,569,C. Mervin Palmer
18,https://openalex.org/W2143121893,4,6,1975,10.1109/tit.1975.1055437,A conditional entropy bound for a pair of disc...,146,H. S. Witsenhausen; A.D. Wyner
23,https://openalex.org/W2040121225,72,7,1972,10.1016/s0019-9958(72)90199-4,A definition of a nonprobabilistic entropy in ...,2241,Aldo de Luca; S. Termini
26,https://openalex.org/W4213350211,32,7,1964,10.1016/s0019-9958(64)90223-2,A formal theory of inductive inference. Part I,1154,Ray J. Solomonoff
27,https://openalex.org/W2135625884,32,9,1964,10.1016/s0019-9958(64)90131-7,A formal theory of inductive inference. Part II,1774,Ray J. Solomonoff
53,https://openalex.org/W2155676967,11,12,1975,10.1109/tit.1975.1055356,A proof of the data compression theorem of Sle...,387,Thomas M. Cover
55,https://openalex.org/W2033020033,12,6,1974,10.1109/tit.1974.1055184,A simple converse for broadcast channels with ...,458,P. Bergmans
56,https://openalex.org/W1993944611,26,13,1965,10.1109/tit.1965.1053730,A simple derivation of the coding theorem and ...,696,Robert G. Gallager


## 2.2 Epistemic Core with Degree Centrality

Here I consider an alternative way of identifying the epistemic core.

Instead of using k-core decomposition, I rank the nodes by degree centrality and take the top 53 publications, so that the size matches that of the epistemic core selected in the previous section. I then compare the two results.

This comparison is not meant to establish that degree centrality is the correct criterion, but to assess how sensitive the identification of the epistemic core is to the analytical method used.

In [8]:
dc = nx.degree_centrality(G)
top53=sorted(dc.items(), key=lambda x: x[1], reverse=True)[:53]


top53_ids = [n[0] for n in top53]

df_shared = df_core[df_core['node_id'].isin(top53_ids)] #create a dataframe with nodes belonging both to the k-core epistemic core and to the degree centrality epistemic core.
df_shared


,node_id,reference_occurrence,degree,year,doi,title,cited_by_count,authors
27,https://openalex.org/W2135625884,32,9,1964,10.1016/s0019-9958(64)90131-7,A formal theory of inductive inference. Part II,1774,Ray J. Solomonoff
53,https://openalex.org/W2155676967,11,12,1975,10.1109/tit.1975.1055356,A proof of the data compression theorem of Sle...,387,Thomas M. Cover
56,https://openalex.org/W1993944611,26,13,1965,10.1109/tit.1965.1053730,A simple derivation of the coding theorem and ...,696,Robert G. Gallager
119,https://openalex.org/W2296441616,20,11,1972,10.1109/tit.1972.1054727,Broadcast channels,1709,Thomas M. Cover
277,https://openalex.org/W4252822198,12,11,1969,10.1007/bf00537520,Information radius,226,Robin Sibson
280,https://openalex.org/W2947000318,103,18,1966,NaN,Information Theory,3905,R. Ash
281,https://openalex.org/W158805393,73,12,1964,10.2307/2343598,Information Theory and Coding.,1088,I. J. Good; N. Abramson
283,https://openalex.org/W1989008275,183,65,1969,10.1049/ep.1969.0319,Information Theory and Reliable Communication,3999,David A. Bell
284,https://openalex.org/W2032558547,452,32,1957,10.1103/physrev.106.620,Information Theory and Statistical Mechanics,12669,E. T. Jaynes
285,https://openalex.org/W4252028749,164,19,1957,10.1103/physrev.108.171,Information Theory and Statistical Mechanics. II,3160,E. T. Jaynes


In [9]:
print ("Number of shared nodes:", len(df_shared))

Number of shared nodes: 19


# 3. Clustering

This section identifies communities in the network.

The purpose of this step is to detect clusters that may correspond, at least approximately, to specialized areas in which information theory was taken up and developed. I first use the Girvan–Newman algorithm, following Herfeld and Doehne’s approach, and then compare the resulting partition with one obtained through the Louvain method. The agreement between the two partitions is measured with the Adjusted Rand Index (ARI).

## 3.1 Girvan-Newman

In [10]:
## First I look at the number and size of componenents of the network. I keep those with 15, 10, 9, 9 as clusters, do not consider the smaller ones. I apply the clustering algorithm to the

component_sizes = sorted((len(c) for c in nx.connected_components(G)), reverse=True)


print("number of components:", len(component_sizes))
print(component_sizes[:20])   # the 20 biggest


number of components: 79
[438, 13, 10, 9, 9, 6, 5, 5, 4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3]


I keep the four connected components composed of 13, 10, 9, and 9 nodes respectively as communities in their own right, and I apply Girvan–Newman only to the largest connected component.

This choice is partly pragmatic and partly interpretive. Very small components are not especially informative for the purposes of the analysis, whereas these larger disconnected components can still be treated as distinct clusters.

In [11]:
#Here I create the graph for the largest connected component

largest_cc_nodes = max(nx.connected_components(G), key=len)
Gcc = G.subgraph(largest_cc_nodes).copy()


### 3.1.1 Apply the Algorithm

In [12]:

"""
this is the function to run the girvan-newman algorithm
it returns:
  - a dataframe with info for each level of the split (Girvan-Newman works by iiteratively breaking edges, so to each level corresponds a possible different partition)
  - the list of partitions (one for each level. A partition in the list is a tuple of set of nodes)
"""
def run_girvan_newman(G):

    comp = nx.community.girvan_newman(G)

    partitions = []
    rows = []

    level = 0
    for communities in comp:
        level += 1

        communities = tuple(sorted(communities, key=len, reverse=True)) #communities are sorted, from the biggest to the smallest

        partitions.append(communities)

        sizes = [len(c) for c in communities]

        row = {                                           #attributes shown in the dataframe for each partition
            "level": level,
            "n_communities": len(communities),
            "community_sizes": sizes,
            "modularity": nx.community.modularity(G, communities)
        }

        rows.append(row)

    results_df = pd.DataFrame(rows)

    return results_df, partitions


In [13]:
results_df, partitions = run_girvan_newman(Gcc)   #apply the function to the largest connected component


df_first_levels =results_df[results_df["n_communities"] < 30]

df_first_levels

,level,n_communities,community_sizes,modularity
0,1,2,"[402, 36]",0.318997
1,2,3,"[339, 63, 36]",0.456187
2,3,4,"[178, 161, 63, 36]",0.679122
3,4,5,"[178, 141, 63, 36, 20]",0.702017
4,5,6,"[178, 128, 63, 36, 20, 13]",0.713017
5,6,7,"[162, 128, 63, 36, 20, 16, 13]",0.730326
6,7,8,"[162, 92, 63, 36, 36, 20, 16, 13]",0.744677
7,8,9,"[155, 92, 63, 36, 36, 20, 16, 13, 7]",0.748692
8,9,10,"[155, 92, 36, 36, 32, 31, 20, 16, 13, 7]",0.752623
9,10,11,"[149, 92, 36, 36, 32, 31, 20, 16, 13, 7, 6]",0.755766



I select the partition with **10 communities** in the largest connected component.

This is not the partition with the highest modularity. However, the partitions with more communities tend to include many very small clusters, which are difficult to interpret as meaningful specialized areas. Since the aim here is not simply to maximize modularity but to obtain a partition that can plausibly be related to fields or areas of application, I retain the 10-community solution. Together with the four disconnected components retained above, this yields **14 communities** in total.



In [14]:
chosen_partition = partitions[8] # I choose the partion at level 9, with 10 communities

"""
    This function assigns a community to the nodes of G:
    - for Gcc, it uses the communities obtained with Girvan-Newman
    - The 4 other biggest components of G become each a community
    - the other receive the value -1

    The function returns a dictionary node_to_community. It also assign the attribute "community" to the nodes of G.


    """

def assign_gn_plus_top_components(
    G,
    gn_partition,
    n_extra_components=4,
    community_attr="community",  #name of the attribute assigned to the nodes of G
    other_value=-1,  #value assigned to the nodes that are not in a community
):

    components_sorted = sorted(nx.connected_components(G), key=len, reverse=True)

    if len(components_sorted) == 0:
        return {}, []

    giant_component = set(components_sorted[0])

    # Check: Girvan-Newman partition must cover precisely largest component.
    gn_nodes = set().union(*gn_partition) if gn_partition else set()
    if gn_nodes != giant_component:
        raise ValueError(
            "Girvan-Newman partition does not coincide with the largest connected component."
        )

    node_to_community = {n: other_value for n in G.nodes()}

    # community of Girvan-Newman
    current_cid = 0
    for comm in gn_partition:
        for node in comm:
            node_to_community[node] = current_cid
        current_cid += 1

    # Other 4 communities
    extra_components = components_sorted[1 : 1 + n_extra_components]
    for comp in extra_components:
        for node in comp:
            node_to_community[node] = current_cid
        current_cid += 1


    nx.set_node_attributes(G, node_to_community, community_attr) #adding the attribute to the nodes of G

    return node_to_community



In [15]:
node_to_community = assign_gn_plus_top_components(
    G,
    gn_partition=chosen_partition,
    n_extra_components=4,
    community_attr="community",
    other_value=-1
)

### 3.1.2 Plot the Graph with Communities

In [16]:
def plot_graph_by_community(
    G,
    community_attr="community",
    title=None,
    height=900,
):
    pos = nx.spring_layout(G, seed=42)

    # edges
    edge_x = []
    edge_y = []
    for u, v in G.edges():
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

    edge_trace = go.Scatter(
        x=edge_x,
        y=edge_y,
        mode="lines",
        hoverinfo="skip",
        line=dict(width=0.4, color="rgba(160,160,160,0.3)")
    )

    # colors
    palette = [
        "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
        "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
        "#393b79", "#637939", "#8c6d31", "#843c39", "#7b4173"
    ]

    node_x = []
    node_y = []
    node_color = []
    hover_text = []

    for n, attrs in G.nodes(data=True):
        x, y = pos[n]
        node_x.append(x)
        node_y.append(y)

        comm = attrs.get(community_attr, -1)
        if comm == -1:
            color = "#d3d3d3"
        else:
            color = palette[comm % len(palette)]
        node_color.append(color)

        txt = f"<b>ID</b>: {n}<br><b>{community_attr}</b>: {comm}<br>"
        for k, v in attrs.items():
            if k != community_attr:
                txt += f"<b>{k}</b>: {v}<br>"
        hover_text.append(txt)

    node_trace = go.Scatter(
        x=node_x,
        y=node_y,
        mode="markers",
        text=hover_text,
        hoverinfo="text",
        marker=dict(
            size=7,
            color=node_color,
            line=dict(width=0.5, color="white")
        )
    )

    fig = go.Figure(data=[edge_trace, node_trace])

    fig.update_layout(
        title=title,
        showlegend=False,
        hovermode="closest",
        margin=dict(l=10, r=10, t=40, b=10),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        height=height
    )

    fig.show()

In [17]:
plot_graph_by_community(
    G,
    community_attr="community",
    title="Graph with communities"
)

Some clusters can be mapped, at least approximately, to specialized fields, while others are harder to interpret in those terms.

This already points to an important limitation of the analysis: the output of community detection does not automatically correspond to well-defined academic fields. In some cases the mapping is plausible, in others it remains unclear or mixed.

0: Overlaps significantly with the epistemic core.

1:  Statistics and probability theory, with noise.

2: Cognitive science, philosophy.

3:  Communication.

4:  ?

5:  Ecology, with noise.

6: Algoritmic complexity, with noise.

7: Fuzzy logic, with noise.

8: Quantum theory.

9:  ?

10: ?

11: ?

12: Data compression.

13: ?




In [18]:
#visualize the communities of the members of the epistemic core

core_nodes_with_communities = []
for node_id, attrs in G_core.nodes(data=True):
    community_id = G.nodes[node_id].get('community', 'N/A')
    core_nodes_with_communities.append({'node_id': node_id, 'title': attrs.get('title'), 'community': community_id})

core_nodes_communities_df = pd.DataFrame(core_nodes_with_communities)
display(core_nodes_communities_df)

,node_id,title,community
0,https://openalex.org/W2043769961,The Wire-Tap Channel,0
1,https://openalex.org/W2033020033,A simple converse for broadcast channels with ...,0
2,https://openalex.org/W2947000318,Information Theory,0
3,https://openalex.org/W2087171258,Mathematical Foundations of Information Theory.,1
4,https://openalex.org/W2154462988,On the Length of Programs for Computing Finite...,6
5,https://openalex.org/W1592482298,Multi-way communication channels,0
6,https://openalex.org/W1981971374,Diversity in some south African diatom associa...,4
7,https://openalex.org/W158805393,Information Theory and Coding.,1
8,https://openalex.org/W1993944611,A simple derivation of the coding theorem and ...,0
9,https://openalex.org/W2017148445,The Capacity Region of a Channel with Two Send...,0


##3.2 Louvain and ARI

### 3.2.1 Louvain

I also apply the Louvain method in order to assess how robust the clustering results are with respect to a different community-detection algorithm.

I experimented with different values of the `resolution` parameter. I report here the result for `resolution = 0.6`, because it yields a number of communities in the largest connected component that is reasonably close to the one obtained with Girvan–Newman (11 vs. 10), and therefore allows for a more meaningful comparison.

In [19]:

# Louvain
communities = nx.community.louvain_communities(
    Gcc,
    resolution=0.6,
    seed=42
)

print("Number of communities:", len(communities))
print("Sizes:", [len(c) for c in communities])

# modularity
mod = nx.community.modularity(Gcc, communities)
print("Modularity:", mod)

Number of communities: 11
Sizes: [48, 14, 6, 99, 18, 31, 7, 26, 54, 36, 99]
Modularity: 0.7776861878826569


### 3.2.2 ARI

I now compare the Girvan–Newman partition and the Louvain partition using the Adjusted Rand Index (ARI).

The ARI provides a measure of similarity between two clustering solutions. Here it is used to assess whether the detected community structure is stable across different algorithms, or whether it is significantly affected by methodological choices.

In [20]:
#first we want two dictionaries, mapping the nodes (only belonging to Gcc) to their communities
gn_partition_filtered = {node: community_id for node, community_id in node_to_community.items() if node in Gcc.nodes()}

louvain_partition_filtered = {}
for i, community_set in enumerate(communities):
    for node in community_set:
        if node in Gcc.nodes():
            louvain_partition_filtered[node] = i



In [21]:
from sklearn.metrics import adjusted_rand_score

# Ensure both dictionaries have the same keys (nodes present in Gcc)
common_nodes = set(gn_partition_filtered.keys()) & set(louvain_partition_filtered.keys())

# Extract community labels in a consistent order
gn_labels = [gn_partition_filtered[node] for node in common_nodes]
louvain_labels = [louvain_partition_filtered[node] for node in common_nodes]

# Calculate Adjusted Rand Index
ari = adjusted_rand_score(gn_labels, louvain_labels)

print(f"Adjusted Rand Index (ARI) between Girvan-Newman and Louvain: {ari:.4f}")

Adjusted Rand Index (ARI) between Girvan-Newman and Louvain: 0.5646


# 4. Roles

This section identifies the roles of **elaborators** and **translators**.

These roles are central to Herfeld and Doehne’s account of scientific innovation. The point of the present analysis is not only to classify nodes, but also to assess whether these roles can be identified in a clear and robust way in the case of information theory.



## 4.1 Elaborators

Elaborators are defined here as nodes that belong to the epistemic core and whose within-community degree centrality is less than or equal to the **q-quantile** of the distribution within their community.

This operationalization is meant to capture the idea that elaborators belong to the epistemic core but are not strongly integrated into a specialized area. Since Herfeld and Doehne do not provide a fully explicit threshold for what counts as “low” centrality, I test different values of `q`.

In [22]:

import numpy as np

"""
The function returns:
- a list of the elaborators
- a dataframe containing all the nodes of the core with infos about degree centrality in their cluster and whether they are elaborators or not

"""

def identify_elaborators(
    G,
    G_core,
    community_attr="community",
    within_deg_attr="within_cluster_degree",
    elaborator_attr="is_elaborator",
    threshold_value=0.5,   #q-quantile
    ignore_communities={-1},
):

    if threshold_value is None or not (0 <= threshold_value <= 1):
        raise ValueError("threshold_value must be between 0 and 1")

    community_map = nx.get_node_attributes(G, community_attr)

    # communities represented in the core
    core_communities = {
        community_map[node]
        for node in G_core.nodes()
        if community_map.get(node) not in ignore_communities
    }


    communities = {}
    for node, cid in community_map.items():
        if cid in ignore_communities or cid not in core_communities:
            continue
        communities.setdefault(cid, []).append(node)

    # within-cluster degree centrality
    within_deg = {}
    for cid, nodes in communities.items():
        H = G.subgraph(nodes)
        dc = nx.degree_centrality(H)
        within_deg.update(dc)

    nx.set_node_attributes(G, within_deg, within_deg_attr)

    # treshold for community
    thresholds = {}
    for cid, nodes in communities.items():
        vals = [within_deg[n] for n in nodes if n in within_deg]
        if not vals:
            continue
        thresholds[cid] = np.quantile(vals, threshold_value)


    elaborators = []
    node_flag = {n: False for n in G.nodes()}
    rows = []

    for node in G_core.nodes():
        cid = community_map.get(node, None)
        deg = within_deg.get(node, None)

        is_elaborator = (
            cid not in ignore_communities
            and cid in thresholds
            and deg is not None
            and deg <= thresholds[cid]
        )

        node_flag[node] = is_elaborator
        if is_elaborator:
            elaborators.append(node)

        rows.append({
            "node": node,
            "community": cid,
            "within_cluster_degree": deg,
            "threshold": thresholds.get(cid, None),
            "is_elaborator": is_elaborator,
        })

    nx.set_node_attributes(G, node_flag, elaborator_attr)

    df = pd.DataFrame(rows)
    df = df.merge(nodes_df[['node_id', 'title']], left_on='node', right_on='node_id', how='left') #add to the dataframe the title of publication
    df.drop(columns=['node_id'], inplace=True) # Drop the redundant node_id column from the merge

    return elaborators, df

In [23]:
elaborators, df = identify_elaborators(G, G_core, threshold_value = 0.5)



df[df["is_elaborator"]]


,node,community,within_cluster_degree,threshold,is_elaborator,title
29,https://openalex.org/W2159977619,6,0.105263,0.184211,True,An Introduction to Information Theory.


When **q = 0.5**, only one node in the epistemic core is identified as an elaborator. The number increases only slightly when the threshold is raised.

This result is important for the broader argument of the paper: in this network, elaborators are difficult to identify in a clear way on the basis of the proposed analytical criterion.

In [24]:
elaborators, df = identify_elaborators(G, G_core, threshold_value = 0.6) #here q= 0.6



df[df["is_elaborator"]]

,node,community,within_cluster_degree,threshold,is_elaborator,title
16,https://openalex.org/W4245152641,7,0.266667,0.266667,True,The concept of a linguistic variable and its a...
18,https://openalex.org/W1976755248,7,0.266667,0.266667,True,Intervall‐wertige Mengen
21,https://openalex.org/W2263958071,7,0.266667,0.266667,True,Quantification method of classification proces...
29,https://openalex.org/W2159977619,6,0.105263,0.231579,True,An Introduction to Information Theory.


## 4.2 Translators

Translators are defined here as nodes whose within-community degree centrality is greater than or equal to the **q-quantile** of the distribution within their community, and that either belong to the epistemic core or are directly connected to it.

This operationalization is meant to capture the idea that translators connect the epistemic core to more specialized areas, thereby enabling the innovation to spread across fields.

In [25]:
import numpy as np

#the function is similar to that for the elaborators

def identify_translators_quantile(
    G,
    G_core,
    community_attr="community",
    translator_attr="is_translator",
    within_deg_attr="within_cluster_degree",
    ignore_communities={-1},
    threshold_value=0.90,
):

    if threshold_value is None or not (0 <= threshold_value <= 1):
        raise ValueError("threshold_value must be between 0 and 1")

    community_map = nx.get_node_attributes(G, community_attr)
    core_nodes = set(G_core.nodes())

    communities = {}
    for node, cid in community_map.items():
        if cid not in ignore_communities:
            communities.setdefault(cid, []).append(node)

    within_deg_all = {}
    for cid, nodes in communities.items():
        H = G.subgraph(nodes)
        dc = nx.degree_centrality(H)
        within_deg_all.update(dc)

    nx.set_node_attributes(G, within_deg_all, within_deg_attr)

    thresholds = {}
    for cid, nodes in communities.items():
        vals = [within_deg_all[n] for n in nodes if n in within_deg_all]
        if not vals:
            continue
        thresholds[cid] = np.quantile(vals, threshold_value)

    translators = []
    translator_flag = {n: False for n in G.nodes()}
    rows = []

    for cid, nodes in communities.items():
        threshold = thresholds.get(cid, None)
        if threshold is None:
            continue

        for node in nodes:
            val = within_deg_all.get(node, None)
            if val is None:
                continue

            in_core = node in core_nodes
            connected_to_core = any(neigh in core_nodes for neigh in G.neighbors(node))

            in_top_quantile = val >= threshold
            is_valid_candidate = in_core or connected_to_core
            is_translator = in_top_quantile and is_valid_candidate

            if is_translator:
                translators.append(node)
                translator_flag[node] = True

            rows.append({
                "node": node,
                "community": cid,
                "within_cluster_degree": val,
                "threshold": threshold,
                "in_top_quantile": in_top_quantile,
                "in_core": in_core,
                "connected_to_core": connected_to_core,
                "is_valid_candidate": is_valid_candidate,
                "is_translator": is_translator,
            })

    nx.set_node_attributes(G, translator_flag, translator_attr)

    df = pd.DataFrame(rows)
    df = df.merge(nodes_df[['node_id', 'title', 'authors']], left_on='node', right_on='node_id', how='left') #add to the dataframe the title of publication and authors
    df.drop(columns=['node_id'], inplace=True) # Drop the redundant node_id column from the merge

    return translators, df

Let us first try with **q = 1**. Under this stricter criterion, we obtain 5 translators.

In [26]:
translators, df_translators = identify_translators_quantile(
    G,
    G_core,
    threshold_value=1
)
df_translators[df_translators["is_translator"]]

,node,community,within_cluster_degree,threshold,in_top_quantile,in_core,connected_to_core,is_valid_candidate,is_translator,title,authors
107,https://openalex.org/W1989008275,0,0.389610,0.389610,True,True,True,True,True,Information Theory and Reliable Communication,David A. Bell
228,https://openalex.org/W2040121225,7,0.466667,0.466667,True,True,True,True,True,A definition of a nonprobabilistic entropy in ...,Aldo de Luca; S. Termini
313,https://openalex.org/W2032558547,1,0.329670,0.329670,True,True,True,True,True,Information Theory and Statistical Mechanics,E. T. Jaynes
402,https://openalex.org/W2104767862,4,0.354839,0.354839,True,True,True,True,True,The Nonconcept of Species Diversity: A Critiqu...,Stuart H. Hurlbert
409,https://openalex.org/W2089442854,6,0.473684,0.473684,True,False,True,True,True,A Machine-Independent Theory of the Complexity...,Manuel Blum


With **q = 0.9**, the number of translators increases to 37.

As discussed in the paper, however, the issue is not only the total number of translators, but also their distribution across communities. In this case, translators are not evenly distributed, which raises difficulties for the generality of the proposed role typology.

In [27]:
translators, df_translators = identify_translators_quantile(
    G,
    G_core,
    threshold_value=0.9
)
df_translators[df_translators["is_translator"]]

,node,community,within_cluster_degree,threshold,in_top_quantile,in_core,connected_to_core,is_valid_candidate,is_translator,title,authors
39,https://openalex.org/W2001935257,0,0.051948,0.045455,True,True,True,True,True,A coding theorem and Rényi's entropy,L. L. Campbell
46,https://openalex.org/W2087362480,0,0.045455,0.045455,True,False,True,True,True,A heuristic discussion of probabilistic decoding,Robert M. Fano
51,https://openalex.org/W2155676967,0,0.077922,0.045455,True,True,True,True,True,A proof of the data compression theorem of Sle...,Thomas M. Cover
53,https://openalex.org/W1993944611,0,0.084416,0.045455,True,True,True,True,True,A simple derivation of the coding theorem and ...,Robert G. Gallager
69,https://openalex.org/W2296441616,0,0.071429,0.045455,True,True,True,True,True,Broadcast channels,Thomas M. Cover
89,https://openalex.org/W1991133427,0,0.058442,0.045455,True,False,True,True,True,Error bounds for convolutional codes and an as...,Andrew J. Viterbi
104,https://openalex.org/W4252822198,0,0.064935,0.045455,True,True,True,True,True,Information radius,Robin Sibson
106,https://openalex.org/W2947000318,0,0.084416,0.045455,True,True,True,True,True,Information Theory,R. Ash
107,https://openalex.org/W1989008275,0,0.389610,0.045455,True,True,True,True,True,Information Theory and Reliable Communication,David A. Bell
112,https://openalex.org/W2128765501,0,0.064935,0.045455,True,False,True,True,True,Low-density parity-check codes,Robert G. Gallager


In [28]:
df_translators[df_translators["is_translator"]].shape

(37, 11)

# 5. Concluding Remark

The notebook documents the computational side of the analysis. Its main result is not simply a classification of nodes, but the emergence of several methodological and interpretive difficulties: the epistemic core depends on analytical choices, community detection is only partly stable, and the roles of elaborators and translators are not straightforwardly recoverable from the topology alone.

For the philosophical significance of these results, see the accompanying paper.